<div style="background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 35px; border-radius: 12px; color: white; margin-bottom: 25px;">
  <h1 style="margin: 0 0 8px 0; font-size: 2em;">🏦 Lab 01: MAF Setup &amp; Architecture</h1>
  <p style="margin: 0; font-size: 1.15em; opacity: 0.92;">Zava Bank Workshop — Hosted Agents with Microsoft Agent Framework</p>
</div>

<div style="background: #f5f5f5; border-left: 4px solid #533483; padding: 16px 20px; border-radius: 6px; margin-bottom: 20px;">

## What We Learn

- Install and validate the **Microsoft Agent Framework (MAF)** SDK
- Understand the **hosted agent architecture** and how it differs from prompt agents
- Load the same Zava Bank data we used in Path A, preparing it for a MAF-based agent

</div>

## 📖 Zava Bank Story

In **Path A** (prompt agents), we built the Zava Bank Client Advisory Agent using the Foundry prompt-agent model. That approach was effective for prototyping — define tools as functions, register them server-side, and let the Foundry runtime handle tool dispatch and orchestration.

Now the team needs a **production path**. The requirements have grown:

- Custom Python logic that runs in-process alongside the model
- Deterministic tool execution controlled by your code, not a black-box runtime
- Full lifecycle control — startup hooks, shutdown cleanup, health checks
- Granular tracing and observability built into the agent class

**Microsoft Agent Framework (MAF)** is how you'd deploy the Zava Bank advisor in production. Instead of registering JSON function schemas and letting the platform call them, you write a Python class that inherits from `BaseAgent` and implement an `on_message` handler. Your code *is* the agent.

## 🔀 Architecture Comparison

| Dimension | Prompt Agent (Path A) | Hosted Agent — MAF (Path B) |
|---|---|---|
| **Where tools execute** | Server-side in Foundry runtime | In-process in your Python container |
| **Customization depth** | Function schemas + system prompt | Full Python class — arbitrary logic |
| **Deployment model** | Registered in Foundry, managed hosting | Container image pushed to Foundry hosted compute |
| **Tracing granularity** | Platform-level spans | Custom spans inside your `on_message` handler |
| **Best for** | Rapid prototyping, simple tool use | Production workloads, complex orchestration |

---
## 1 · Install &amp; Validate MAF SDK

In [ ]:
# Install the Microsoft Agent Framework SDK
%pip install azure-ai-agentserver-agentframework>=1.0.0b16 --quiet

In [ ]:
import importlib, sys

required = [
    "azure.ai.agentserver.agentframework",
    "azure.identity",
    "azure.ai.projects",
    "pandas",
    "dotenv",
]

for mod in required:
    try:
        importlib.import_module(mod)
        print(f"  ✅ {mod}")
    except ImportError:
        print(f"  ❌ {mod} — run pip install for this package")

print(f"\nPython {sys.version}")

---
## 2 · Environment &amp; Clients

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Walk up to find the .env file at the repo root
notebook_dir = Path.cwd()
env_path = notebook_dir
while env_path != env_path.parent:
    if (env_path / ".env").exists():
        break
    env_path = env_path.parent

load_dotenv(env_path / ".env", override=True)

PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT", "")
MODEL_DEPLOYMENT = os.getenv("MODEL_DEPLOYMENT_NAME", "gpt-4o")

print(f"Project endpoint : {PROJECT_ENDPOINT[:60]}..." if len(PROJECT_ENDPOINT) > 60 else f"Project endpoint : {PROJECT_ENDPOINT}")
print(f"Model deployment : {MODEL_DEPLOYMENT}")

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)
print("✅ AIProjectClient ready")

---
## 3 · Load Zava Bank Data

Same three CSVs we used in Path A — client portfolios, market indicators, and risk metrics.

In [ ]:
import pandas as pd

DATA_DIR = notebook_dir.parents[1] / "data" / "zava-bank"

portfolios_df = pd.read_csv(DATA_DIR / "client_portfolios.csv")
market_df     = pd.read_csv(DATA_DIR / "market_data.csv")
risk_df       = pd.read_csv(DATA_DIR / "risk_metrics.csv")

print(f"Portfolios : {len(portfolios_df)} rows  — {portfolios_df['client_id'].nunique()} clients")
print(f"Market data: {len(market_df)} indicators")
print(f"Risk metrics: {len(risk_df)} client profiles")

In [ ]:
portfolios_df.head()

In [ ]:
risk_df.head()

In [ ]:
market_df.head()

---
## 4 · The MAF Mental Model

<div style="background: #f5f5f5; border-left: 4px solid #e94560; padding: 16px 20px; border-radius: 6px; margin: 12px 0;">

**Key concept** — In MAF, your agent is a Python class that inherits from `BaseAgent`. The core contract is a single method:

```python
class MyAgent(BaseAgent):
    async def on_message(self, context: AgentContext, message: str) -> str:
        # Your logic here
        ...
```

The `AgentContext` gives you access to:
- `context.invoke_model(...)` — call the LLM with a system prompt and user message
- `context.session` — manage conversation state across turns
- `context.logger` — structured logging tied to the agent lifecycle

Everything that was implicit in a prompt agent (tool dispatch, system prompt injection, response formatting) becomes **explicit Python code** that you own.

</div>

In [ ]:
# Quick verification that the MAF import works
from azure.ai.agentserver.agentframework import BaseAgent, AgentContext

print(f"BaseAgent  : {BaseAgent}")
print(f"AgentContext: {AgentContext}")
print("\n✅ MAF SDK imported — ready to build agents in Lab 02")

---

<div style="background: linear-gradient(135deg, #0f3460, #533483); padding: 20px 25px; border-radius: 10px; color: white; margin-top: 25px;">

## ✅ Lab 01 Summary

We installed the MAF SDK, loaded the Zava Bank datasets, and established the mental model for hosted agents.

**What makes MAF different from prompt agents:**
- Your agent is a **Python class**, not a JSON definition
- Tools execute **in your process**, not server-side
- You control the full lifecycle — `on_message` is your entry point
- Tracing is granular because you instrument your own code

**Next → Lab 02**: Build the `ZavaBankAdvisor` class and handle real advisory queries.

</div>